# EditGuard — controlled operations benchmark

This notebook profiles and evaluates the implemented five-condition
comparison. No training runs. The author ReasonKE checkpoint and official
Qwen instruction checkpoint are loaded sequentially on one T4, never together.
Keep a separate runtime from the chat demo so no other model occupies VRAM.

The dataset contains 48 isolated edit lineages split before wording
augmentation. Final inference requires a source/data/profile-bound frozen
protocol. All wrong, abstained, failed and unexecuted outcomes are retained.
A wrong answer is not retried. Runtime storage is temporary: export receipts
after each stage. No published accuracy number is substituted for our run.


In [ ]:
from pathlib import Path
import json, os, secrets, shutil, subprocess, sys, time, urllib.request
from datetime import datetime, timezone
import torch

SOURCE_COMMIT = '1548db9b1abcaf098fc11dc74f4ad6e3a9a475a6'
REPOSITORY_URL = 'https://github.com/ShivamRaghav7/verity.git'
REPO = Path('/content/editguard-reasonke')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before downloading the model.')
gpu = torch.cuda.get_device_properties(0)
inventory = {'started_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': gpu.name, 'gpu_bytes': gpu.total_memory,
    'free_disk_bytes': shutil.disk_usage('/content').free,
    'source_commit': SOURCE_COMMIT}
print(json.dumps(inventory, indent=2))
if inventory['free_disk_bytes'] < 24 * 1024**3:
    raise RuntimeError('At least 24 GiB of free runtime disk is required for this download candidate.')


In [ ]:
if REPO.exists():
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != SOURCE_COMMIT:
        raise RuntimeError('Existing checkout differs; preserve it and use a fresh runtime.')
else:
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPOSITORY_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
os.chdir(REPO)
VENV = Path('/content/editguard-reasonke-venv')
# Also repairs pyvenv.cfg after an interrupted ensurepip-based creation.
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
for command in ([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--index-url', 'https://pypi.org/simple', '-r', 'requirements-reasonke.txt'],
                [PYTHON, '-m', 'unittest', 'tests.test_reasonke_runtime', 'tests.test_chat_reasonke', '-q']):
    completed = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode or 'unittest' not in command:
        print(completed.stdout)
    else:
        print('Software-contract tests passed. Expected fixture failures are test inputs, not model results.')
    completed.check_returncode()
print('Contract tests passed; they do not establish model answer accuracy.')


In [ ]:
# Finite profile: 12 actual requests per checkpoint, sequentially.
# Both source tensors are about 15.23 GB; a fresh second checkpoint needs disk.
DIRECTORY = REPO / 'artifacts/operations'
DIRECTORY.mkdir(parents=True, exist_ok=True)
PROFILE_LOG = DIRECTORY / 'profile-pipeline.log'
if 'PROFILE_JOB' in globals() and PROFILE_JOB.poll() is None:
    print('The existing profile pipeline is running. Do not launch another.')
else:
    handle = PROFILE_LOG.open('a', encoding='utf-8')
    PROFILE_JOB = subprocess.Popen([PYTHON, '-u', '-m', 'scripts.run_operations_pipeline',
        '--phase', 'profile', '--directory', str(DIRECTORY)], cwd=REPO,
        env=dict(os.environ, HF_HUB_DOWNLOAD_TIMEOUT='180', TOKENIZERS_PARALLELISM='false'),
        stdout=handle, stderr=subprocess.STDOUT)
    handle.close()
    print('One profile pipeline launched; 24 total assigned model generations.')


In [ ]:
print(PROFILE_LOG.read_text(errors='replace')[-3000:])
for name in ('reasonke', 'instruction'):
    path = DIRECTORY / ('profile-' + name + '.json')
    if path.exists():
        receipt = json.loads(path.read_text())
        done = [row for row in receipt['records'] if row['outcome'] != 'unexecuted']
        print(name, 'status:', receipt['status'], 'recorded:', len(done), 'assigned:', len(receipt['records']))
        if done: print('Latest:', done[-1]['id'], done[-1]['outcome'], done[-1].get('pipeline_seconds'))
    else: print(name, 'not started yet')
if PROFILE_JOB.poll() is not None:
    print('Pipeline exit:', PROFILE_JOB.returncode)


## Development and freeze

Inspect profiles and the measured resource budget first. Run development
through the same pipeline with phase `development` (120 assigned requests).
Use development only for prompt/retrieval/scoring choices. If source or
settings change, profiles and protocol identities must be refreshed under
a declared revised run; old receipts stay preserved.

Freeze only after settings are fixed. The protocol records actual profile
means/maxima and extrapolations for 784 ReasonKE and 528 instruction final
requests, including 16 oracle-support questions per model. Those forecasts
are uncertain and exclude cold downloads/loading. Review them before the
substantial final command. No final test tuning or answer-based retries.


In [ ]:
# This cell only seals the fixed protocol; it performs zero generations.
PROTOCOL = DIRECTORY / 'protocol.json'
if PROTOCOL.exists():
    print('Existing protocol preserved:', PROTOCOL)
else:
    completed = subprocess.run([PYTHON, '-m', 'editguard_benchmark.run', '--freeze-profiles',
        str(DIRECTORY/'profile-reasonke.json'), str(DIRECTORY/'profile-instruction.json'),
        '--output', str(PROTOCOL)], cwd=REPO, text=True, capture_output=True)
    print(completed.stdout, completed.stderr)
    completed.check_returncode()


In [ ]:
# Explicit finite final stage. Enable after reviewing the profile-grounded budget.
RUN_FINAL = False
if not RUN_FINAL:
    print('Final generation is disabled until the measured budget is reviewed.')
elif 'FINAL_JOB' in globals() and FINAL_JOB.poll() is None:
    print('The existing final pipeline is running.')
else:
    handle = (DIRECTORY/'final-pipeline.log').open('a', encoding='utf-8')
    FINAL_JOB = subprocess.Popen([PYTHON, '-u', '-m', 'scripts.run_operations_pipeline',
        '--phase', 'final', '--protocol', str(PROTOCOL), '--directory', str(DIRECTORY)],
        cwd=REPO, stdout=handle, stderr=subprocess.STDOUT)
    handle.close()
    print('Final pipeline launched; exactly 1,312 assigned requests, no training.')


In [ ]:
# Run after the two final jobs have ended; all outcomes remain in denominators.
RESULTS = REPO/'reports/reasonke/operations-final/results.json'
completed = subprocess.run([PYTHON, '-m', 'editguard_benchmark.metrics',
    str(DIRECTORY/'final-reasonke.json'), str(DIRECTORY/'final-instruction.json'),
    '--output', str(RESULTS)], cwd=REPO, text=True, capture_output=True)
print(completed.stdout, completed.stderr)
completed.check_returncode()


In [ ]:
# Download bounded raw receipts and inputs. No weights, chat database or tokens.
import zipfile
bundle = Path('/content/editguard-operations-evidence.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(DIRECTORY.glob('*.json')):
        archive.write(path, str(path.relative_to(REPO)))
    for relative in ('configs/operations-evaluation.json', 'configs/instruction-model.json',
                     'configs/reasonke-model.json', 'data/reasonke/operations-v1/manifest.json'):
        archive.write(REPO/relative, relative)
    results = REPO/'reports/reasonke/operations-final/results.json'
    if results.exists(): archive.write(results, str(results.relative_to(REPO)))
from google.colab import files
files.download(str(bundle))
